# Raw Data Inventory — Bronze Layer Planning

As per your assignment context, raw data is located at:
`/Workspace/Users/ranjanayush585@gmail.com/aistra_data/ingest_data/data/`


---

## Feed Inventory

### 1. POS Transactions
**Source Path:** `raw/pos_transactions/`  
**Format:** Parquet  
**Partitioning:** `ingest_date` (4368 partitions)  
**Row Count:** ~4M rows  
**Columns:** 16 columns  
- `txn_id, txn_line_no, basket_id, outlet_code, channel, sku_code`
- `event_ts` (StringType), `qty, unit_price, discount_amount, tax_amount`
- `payment_mode, till_id, cashier_id, promo_code, source_file`

**Key Characteristics:**
- Append-only feed
- `qty` is 50% null (known data quality issue)
- `event_ts` is stored as StringType (requires type conversion)

**Recommended Bronze Table:** `bronze.pos_transactions`

---

### 2. Reefer Telemetry
**Source Path:** `raw/reefer_telemetry/`  
**Format:** Parquet  
**Partitioning:** `dt` (4368 partitions)  
**Row Count:** ~3.7M rows  
**Columns:** 15 columns  
- `device_id, telemetry_vendor, firmware_version, vehicle_registration`
- `route_code, warehouse_code, gateway_id, reading_ts`
- `temp_value, temp_unit, humidity_pct, door_open_flag`
- `gps_lat, gps_lon, battery_pct`

**Key Characteristics:**
- Append-only feed
- Corrupt file exists at `dt=2025-07-14/part-00000.parquet`
- `temp_unit` is 8% null
- Requires partition-by-partition read on serverless (no ignoreCorruptFiles)

**Recommended Bronze Table:** `bronze.reefer_telemetry`

---

### 3. WMS Scan Events
**Source Path:** `raw/wms_scan_events/`  
**Format:** Parquet  
**Partitioning:** `dt` (2184 partitions)  
**Row Count:** ~1.5M rows  
**Columns:** 12 columns  
- `scan_id, warehouse_code, event_type, order_number, sku_code`
- `batch_id, qty_cases, pallet_id, dock_door, operator_id`
- `handheld_device, event_ts`

**Key Characteristics:**
- Append-only feed
- No nulls detected
- Clean data quality

**Recommended Bronze Table:** `bronze.wms_scan_events`

---

### 4. ERP CDC — Outlet Master
**Source Path:** `raw/erp_cdc/outlet_master/`  
**Format:** Parquet  
**Partitioning:** `extract_date` (546 partitions)  
**Row Count:** Variable (CDC snapshots)  
**Columns:** 14 columns  
- CDC metadata: `__op, __op_ts, __seq`
- Business columns: `outlet_code, outlet_name, channel, outlet_format`
- `city, route_code, warehouse_code, credit_limit, credit_terms_days`
- `gst_number, status`

**Key Characteristics:**
- CDC feed with operation types (Insert/Update/Delete)
- NOT covered in manifest
- Business key: `outlet_code`

**Recommended Bronze Table:** `bronze.erp_outlet_master`

---

### 5. ERP CDC — Product Master
**Source Path:** `raw/erp_cdc/product_master/`  
**Format:** Parquet  
**Partitioning:** `extract_date` (542 partitions)  
**Row Count:** Variable (CDC snapshots)  
**Columns:** 14 columns  
- CDC metadata: `__op, __op_ts, __seq`
- Business columns: `sku_code, product_name, category, brand`
- `case_pack, mrp, list_price, gst_rate_pct, shelf_life_days`
- `is_chilled, status`

**Key Characteristics:**
- CDC feed with operation types
- **4 missing partition dates** identified
- NOT covered in manifest
- Business key: `sku_code`

**Recommended Bronze Table:** `bronze.erp_product_master`

---

### 6. ERP CDC — Sales Order Header
**Source Path:** `raw/erp_cdc/sales_order_header/`  
**Format:** Parquet  
**Partitioning:** `extract_date` (546 partitions)  
**Row Count:** Variable (CDC snapshots)  
**Columns:** 15 columns  
- CDC metadata: `__op, __op_ts, __seq`
- Business columns: `order_number, outlet_code, warehouse_code, route_code`
- `order_date, requested_delivery_date, order_status, line_count`
- `order_value_gross, discount_amount, tax_amount, source_system`

**Key Characteristics:**
- CDC feed with operation types
- NOT covered in manifest
- Business key: `order_number`

**Recommended Bronze Table:** `bronze.erp_sales_order_header`

---

## Bronze Table Summary

| Feed | Bronze Table Name | Format | Partition Strategy |
|------|-------------------|--------|--------------------|
| pos_transactions | bronze.pos_transactions | Delta | By `ingest_date` |
| reefer_telemetry | bronze.reefer_telemetry | Delta | By `dt` |
| wms_scan_events | bronze.wms_scan_events | Delta | By `dt` |
| erp_cdc/outlet_master | bronze.erp_outlet_master | Delta | By `extract_date` |
| erp_cdc/product_master | bronze.erp_product_master | Delta | By `extract_date` |
| erp_cdc/sales_order_header | bronze.erp_sales_order_header | Delta | By `extract_date` |

---

## Next Steps

1. Create Bronze schema in Unity Catalog
2. Implement incremental ingestion logic for each feed
3. Handle corrupt files (reefer_telemetry) with partition-level reads
4. Add audit columns (ingestion timestamp, source file, etc.)
5. Set up idempotency checks for CDC feeds

In [0]:
%sql
-- Create Bronze schema in aistra_ayush catalog
-- Idempotent: safe to rerun

-- Verify catalog exists
USE CATALOG aistra_ayush;

-- Create Bronze schema if it doesn't exist
-- Unity Catalog manages the location automatically
CREATE SCHEMA IF NOT EXISTS bronze
COMMENT 'Bronze layer - raw data ingestion with minimal transformation';

-- Verify schema creation
SHOW SCHEMAS IN aistra_ayush;

In [0]:
%sql
-- Verify Bronze schema details
DESCRIBE SCHEMA EXTENDED aistra_ayush.bronze;

## Bronze Ingestion — POS Transactions

**Source:** `/Workspace/Users/ranjanayush585@gmail.com/aistra-ayush/ingest_data/data/raw/pos_transactions/`  
**Target:** `aistra_ayush.bronze.pos_transactions`  
**Strategy:** Incremental append with file-level deduplication

### Rerun Protection Mechanism:
1. Track source files using `input_file_name()` in `_source_file` column
2. Query existing Bronze table for already-ingested files
3. Only read and append new files that haven't been processed
4. If Bronze table doesn't exist, process all files (initial load)

### Metadata Columns Added:
- `_source_file` — Full path of source Parquet file
- `_ingestion_run_id` — UUID for this ingestion run (groups files processed together)
- `_ingested_at` — Timestamp when record was ingested into Bronze

In [0]:
from pyspark.sql import functions as F
import uuid

RAW_POS_PATH = "/Workspace/Users/ranjanayush585@gmail.com/aistra_data/ingest_data/data/raw/pos_transactions/"
BRONZE_TABLE = "aistra_ayush.bronze.pos_transactions"

run_id = str(uuid.uuid4())

# Identify all source files currently available
source_files = (
    spark.read
    .format("binaryFile")
    .option("recursiveFileLookup", "true")
    .load(RAW_POS_PATH)
    .select(
        F.col("path").alias("_source_file")
    )
)

# Create Bronze table if it does not exist
if not spark.catalog.tableExists(BRONZE_TABLE):
    (
        spark.read
        .parquet(RAW_POS_PATH)
        .limit(0)
        .withColumn("_source_file", F.lit(None).cast("string"))
        .withColumn("_ingestion_run_id", F.lit(None).cast("string"))
        .withColumn("_ingested_at", F.lit(None).cast("timestamp"))
        .write
        .format("delta")
        .saveAsTable(BRONZE_TABLE)
    )

# Find files that have already been ingested
existing_files = (
    spark.table(BRONZE_TABLE)
    .select("_source_file")
    .distinct()
)

new_files = source_files.join(
    existing_files,
    on="_source_file",
    how="left_anti"
)

new_file_count = new_files.count()

print(f"New source files found: {new_file_count}")

if new_file_count > 0:

    # Read raw POS data and add metadata
    raw_df = (
        spark.read
        .option("recursiveFileLookup", "true")
        .parquet(RAW_POS_PATH)
        .select("*", F.col("_metadata.file_path").alias("_source_file"))
    )

    # Keep only files not previously ingested
    new_df = (
        raw_df.join(
            new_files.select("_source_file"),
            on="_source_file",
            how="inner"
        )
        .withColumn("_ingestion_run_id", F.lit(run_id))
        .withColumn("_ingested_at", F.current_timestamp())
    )

    (
        new_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(BRONZE_TABLE)
    )

    print(f"Ingested {new_file_count} new files.")

else:
    print("No new files. Bronze table is already up to date.")

In [0]:
%sql
SELECT *
FROM aistra_ayush.bronze.pos_transactions
LIMIT 10;

In [0]:
%sql
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT _source_file) AS source_files
FROM aistra_ayush.bronze.pos_transactions;

## Bronze Ingestion — Reefer Telemetry

**Source:** `/Workspace/Users/ranjanayush585@gmail.com/aistra_data/ingest_data/data/raw/reefer_telemetry/`  
**Target:** `aistra_ayush.bronze.reefer_telemetry`  
**Strategy:** Incremental append with file-level deduplication

### Special Handling:
- **Corrupt file exists** at `dt=2025-07-14/part-00000.parquet`
- Serverless compute doesn't support `ignoreCorruptFiles`
- Use try-except to skip corrupt files gracefully

### Metadata Columns Added:
- `_source_file` — Full path of source Parquet file
- `_ingestion_run_id` — UUID for this ingestion run
- `_ingested_at` — Timestamp when record was ingested into Bronze

In [0]:
from pyspark.sql import functions as F
import uuid

RAW_REEFER_PATH = "/Workspace/Users/ranjanayush585@gmail.com/aistra_data/ingest_data/data/raw/reefer_telemetry/"
BRONZE_TABLE = "aistra_ayush.bronze.reefer_telemetry"

run_id = str(uuid.uuid4())

# Identify all source files currently available
source_files = (
    spark.read
    .format("binaryFile")
    .option("recursiveFileLookup", "true")
    .load(RAW_REEFER_PATH)
    .select(
        F.col("path").alias("_source_file")
    )
)

# Create Bronze table if it does not exist
if not spark.catalog.tableExists(BRONZE_TABLE):
    (
        spark.read
        .parquet(RAW_REEFER_PATH)
        .limit(0)
        .withColumn("_source_file", F.lit(None).cast("string"))
        .withColumn("_ingestion_run_id", F.lit(None).cast("string"))
        .withColumn("_ingested_at", F.lit(None).cast("timestamp"))
        .write
        .format("delta")
        .saveAsTable(BRONZE_TABLE)
    )

# Find files that have already been ingested
existing_files = (
    spark.table(BRONZE_TABLE)
    .select("_source_file")
    .distinct()
)

new_files = source_files.join(
    existing_files,
    on="_source_file",
    how="left_anti"
)

new_file_count = new_files.count()

print(f"New source files found: {new_file_count}")

if new_file_count > 0:
    # Filter out the known corrupt file
    # Serverless doesn't support ignoreCorruptFiles, so we exclude it explicitly
    corrupt_file_pattern = "dt=2025-07-14/part-00000.parquet"
    
    files_to_ingest = new_files.filter(
        ~F.col("_source_file").contains(corrupt_file_pattern)
    )
    
    filtered_count = files_to_ingest.count()
    skipped_count = new_file_count - filtered_count
    
    if skipped_count > 0:
        print(f"Skipping {skipped_count} corrupt file(s)")
    
    if filtered_count > 0:
        # Read raw reefer data
        raw_df = (
            spark.read
            .option("recursiveFileLookup", "true")
            .parquet(RAW_REEFER_PATH)
            .select("*", F.col("_metadata.file_path").alias("_source_file"))
        )
        
        # Keep only files not previously ingested (and not corrupt)
        new_df = (
            raw_df.join(
                files_to_ingest.select("_source_file"),
                on="_source_file",
                how="inner"
            )
            .withColumn("_ingestion_run_id", F.lit(run_id))
            .withColumn("_ingested_at", F.current_timestamp())
        )
        
        (
            new_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(BRONZE_TABLE)
        )
        
        print(f"Ingested {filtered_count} new files.")
    else:
        print("All new files are corrupt. No data ingested.")
else:
    print("No new files. Bronze table is already up to date.")

In [0]:
%sql
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT _source_file) AS source_files
FROM aistra_ayush.bronze.reefer_telemetry;

## Bronze Ingestion — WMS Scan Events

**Source:** `/Workspace/Users/ranjanayush585@gmail.com/aistra_data/ingest_data/data/raw/wms_scan_events/`  
**Target:** `aistra_ayush.bronze.wms_scan_events`  
**Strategy:** Incremental append with file-level deduplication

### Key Characteristics:
- Clean data quality (no nulls detected)
- Straightforward append-only feed

### Metadata Columns Added:
- `_source_file` — Full path of source Parquet file
- `_ingestion_run_id` — UUID for this ingestion run
- `_ingested_at` — Timestamp when record was ingested into Bronze

In [0]:
from pyspark.sql import functions as F
import uuid

RAW_WMS_PATH = "/Workspace/Users/ranjanayush585@gmail.com/aistra_data/ingest_data/data/raw/wms_scan_events/"
BRONZE_TABLE = "aistra_ayush.bronze.wms_scan_events"

run_id = str(uuid.uuid4())

# Identify all source files currently available
source_files = (
    spark.read
    .format("binaryFile")
    .option("recursiveFileLookup", "true")
    .load(RAW_WMS_PATH)
    .select(
        F.col("path").alias("_source_file")
    )
)

# Create Bronze table if it does not exist
if not spark.catalog.tableExists(BRONZE_TABLE):
    (
        spark.read
        .parquet(RAW_WMS_PATH)
        .limit(0)
        .withColumn("_source_file", F.lit(None).cast("string"))
        .withColumn("_ingestion_run_id", F.lit(None).cast("string"))
        .withColumn("_ingested_at", F.lit(None).cast("timestamp"))
        .write
        .format("delta")
        .saveAsTable(BRONZE_TABLE)
    )

# Find files that have already been ingested
existing_files = (
    spark.table(BRONZE_TABLE)
    .select("_source_file")
    .distinct()
)

new_files = source_files.join(
    existing_files,
    on="_source_file",
    how="left_anti"
)

new_file_count = new_files.count()

print(f"New source files found: {new_file_count}")

if new_file_count > 0:
    # Read raw WMS data and add metadata
    raw_df = (
        spark.read
        .option("recursiveFileLookup", "true")
        .parquet(RAW_WMS_PATH)
        .select("*", F.col("_metadata.file_path").alias("_source_file"))
    )
    
    # Keep only files not previously ingested
    new_df = (
        raw_df.join(
            new_files.select("_source_file"),
            on="_source_file",
            how="inner"
        )
        .withColumn("_ingestion_run_id", F.lit(run_id))
        .withColumn("_ingested_at", F.current_timestamp())
    )
    
    (
        new_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(BRONZE_TABLE)
    )
    
    print(f"Ingested {new_file_count} new files.")
else:
    print("No new files. Bronze table is already up to date.")

In [0]:
%sql
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT _source_file) AS source_files
FROM aistra_ayush.bronze.wms_scan_events;

## Bronze Ingestion — ERP CDC Feeds

**Strategy:** Incremental append with file-level deduplication  
**Special Handling:** Preserve CDC metadata columns (`__op`, `__op_ts`, `__seq`)

### Feed 1: Outlet Master
**Source:** `/Workspace/Users/ranjanayush585@gmail.com/aistra_data/ingest_data/data/raw/erp_cdc/outlet_master/`  
**Target:** `aistra_ayush.bronze.erp_outlet_master`  
**Business Key:** `outlet_code`

### Feed 2: Product Master
**Source:** `/Workspace/Users/ranjanayush585@gmail.com/aistra_data/ingest_data/data/raw/erp_cdc/product_master/`  
**Target:** `aistra_ayush.bronze.erp_product_master`  
**Business Key:** `sku_code`  
**Note:** 4 missing partition dates identified

### Feed 3: Sales Order Header
**Source:** `/Workspace/Users/ranjanayush585@gmail.com/aistra_data/ingest_data/data/raw/erp_cdc/sales_order_header/`  
**Target:** `aistra_ayush.bronze.erp_sales_order_header`  
**Business Key:** `order_number`

### Metadata Columns Added:
- `_source_file` — Full path of source Parquet file
- `_ingestion_run_id` — UUID for this ingestion run
- `_ingested_at` — Timestamp when record was ingested into Bronze

In [0]:
from pyspark.sql import functions as F
import uuid

RAW_OUTLET_PATH = "/Workspace/Users/ranjanayush585@gmail.com/aistra_data/ingest_data/data/raw/erp_cdc/outlet_master/"
BRONZE_TABLE = "aistra_ayush.bronze.erp_outlet_master"

run_id = str(uuid.uuid4())

# Identify all source files currently available
source_files = (
    spark.read
    .format("binaryFile")
    .option("recursiveFileLookup", "true")
    .load(RAW_OUTLET_PATH)
    .select(
        F.col("path").alias("_source_file")
    )
)

# Create Bronze table if it does not exist
if not spark.catalog.tableExists(BRONZE_TABLE):
    (
        spark.read
        .parquet(RAW_OUTLET_PATH)
        .limit(0)
        .withColumn("_source_file", F.lit(None).cast("string"))
        .withColumn("_ingestion_run_id", F.lit(None).cast("string"))
        .withColumn("_ingested_at", F.lit(None).cast("timestamp"))
        .write
        .format("delta")
        .saveAsTable(BRONZE_TABLE)
    )

# Find files that have already been ingested
existing_files = (
    spark.table(BRONZE_TABLE)
    .select("_source_file")
    .distinct()
)

new_files = source_files.join(
    existing_files,
    on="_source_file",
    how="left_anti"
)

new_file_count = new_files.count()

print(f"New source files found: {new_file_count}")

if new_file_count > 0:
    # Read raw ERP CDC data (preserve CDC columns: __op, __op_ts, __seq)
    raw_df = (
        spark.read
        .option("recursiveFileLookup", "true")
        .parquet(RAW_OUTLET_PATH)
        .select("*", F.col("_metadata.file_path").alias("_source_file"))
    )
    
    # Keep only files not previously ingested
    new_df = (
        raw_df.join(
            new_files.select("_source_file"),
            on="_source_file",
            how="inner"
        )
        .withColumn("_ingestion_run_id", F.lit(run_id))
        .withColumn("_ingested_at", F.current_timestamp())
    )
    
    (
        new_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(BRONZE_TABLE)
    )
    
    print(f"Ingested {new_file_count} new files.")
else:
    print("No new files. Bronze table is already up to date.")

In [0]:
%sql
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT _source_file) AS source_files,
    COUNT(DISTINCT outlet_code) AS unique_outlets
FROM aistra_ayush.bronze.erp_outlet_master;

In [0]:
from pyspark.sql import functions as F
import uuid

RAW_PRODUCT_PATH = "/Workspace/Users/ranjanayush585@gmail.com/aistra_data/ingest_data/data/raw/erp_cdc/product_master/"
BRONZE_TABLE = "aistra_ayush.bronze.erp_product_master"

run_id = str(uuid.uuid4())

# Identify all source files currently available
source_files = (
    spark.read
    .format("binaryFile")
    .option("recursiveFileLookup", "true")
    .load(RAW_PRODUCT_PATH)
    .select(
        F.col("path").alias("_source_file")
    )
)

# Create Bronze table if it does not exist
if not spark.catalog.tableExists(BRONZE_TABLE):
    (
        spark.read
        .parquet(RAW_PRODUCT_PATH)
        .limit(0)
        .withColumn("_source_file", F.lit(None).cast("string"))
        .withColumn("_ingestion_run_id", F.lit(None).cast("string"))
        .withColumn("_ingested_at", F.lit(None).cast("timestamp"))
        .write
        .format("delta")
        .saveAsTable(BRONZE_TABLE)
    )

# Find files that have already been ingested
existing_files = (
    spark.table(BRONZE_TABLE)
    .select("_source_file")
    .distinct()
)

new_files = source_files.join(
    existing_files,
    on="_source_file",
    how="left_anti"
)

new_file_count = new_files.count()

print(f"New source files found: {new_file_count}")

if new_file_count > 0:
    # Read raw ERP CDC data (preserve CDC columns: __op, __op_ts, __seq)
    raw_df = (
        spark.read
        .option("recursiveFileLookup", "true")
        .parquet(RAW_PRODUCT_PATH)
        .select("*", F.col("_metadata.file_path").alias("_source_file"))
    )
    
    # Keep only files not previously ingested
    new_df = (
        raw_df.join(
            new_files.select("_source_file"),
            on="_source_file",
            how="inner"
        )
        .withColumn("_ingestion_run_id", F.lit(run_id))
        .withColumn("_ingested_at", F.current_timestamp())
    )
    
    (
        new_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(BRONZE_TABLE)
    )
    
    print(f"Ingested {new_file_count} new files.")
else:
    print("No new files. Bronze table is already up to date.")

In [0]:
%sql
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT _source_file) AS source_files,
    COUNT(DISTINCT sku_code) AS unique_skus
FROM aistra_ayush.bronze.erp_product_master;

In [0]:
from pyspark.sql import functions as F
import uuid

RAW_ORDER_PATH = "/Workspace/Users/ranjanayush585@gmail.com/aistra_data/ingest_data/data/raw/erp_cdc/sales_order_header/"
BRONZE_TABLE = "aistra_ayush.bronze.erp_sales_order_header"

run_id = str(uuid.uuid4())

# Identify all source files currently available
source_files = (
    spark.read
    .format("binaryFile")
    .option("recursiveFileLookup", "true")
    .load(RAW_ORDER_PATH)
    .select(
        F.col("path").alias("_source_file")
    )
)

# Create Bronze table if it does not exist
if not spark.catalog.tableExists(BRONZE_TABLE):
    (
        spark.read
        .parquet(RAW_ORDER_PATH)
        .limit(0)
        .withColumn("_source_file", F.lit(None).cast("string"))
        .withColumn("_ingestion_run_id", F.lit(None).cast("string"))
        .withColumn("_ingested_at", F.lit(None).cast("timestamp"))
        .write
        .format("delta")
        .saveAsTable(BRONZE_TABLE)
    )

# Find files that have already been ingested
existing_files = (
    spark.table(BRONZE_TABLE)
    .select("_source_file")
    .distinct()
)

new_files = source_files.join(
    existing_files,
    on="_source_file",
    how="left_anti"
)

new_file_count = new_files.count()

print(f"New source files found: {new_file_count}")

if new_file_count > 0:
    # Read raw ERP CDC data (preserve CDC columns: __op, __op_ts, __seq)
    raw_df = (
        spark.read
        .option("recursiveFileLookup", "true")
        .parquet(RAW_ORDER_PATH)
        .select("*", F.col("_metadata.file_path").alias("_source_file"))
    )
    
    # Keep only files not previously ingested
    new_df = (
        raw_df.join(
            new_files.select("_source_file"),
            on="_source_file",
            how="inner"
        )
        .withColumn("_ingestion_run_id", F.lit(run_id))
        .withColumn("_ingested_at", F.current_timestamp())
    )
    
    (
        new_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(BRONZE_TABLE)
    )
    
    print(f"Ingested {new_file_count} new files.")
else:
    print("No new files. Bronze table is already up to date.")

In [0]:
%sql
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT _source_file) AS source_files,
    COUNT(DISTINCT order_number) AS unique_orders
FROM aistra_ayush.bronze.erp_sales_order_header;

In [0]:
%sql
-- Summary of all Bronze tables
SELECT 
    'pos_transactions' AS feed,
    COUNT(*) AS rows,
    COUNT(DISTINCT _source_file) AS files
FROM aistra_ayush.bronze.pos_transactions

UNION ALL

SELECT 
    'reefer_telemetry' AS feed,
    COUNT(*) AS rows,
    COUNT(DISTINCT _source_file) AS files
FROM aistra_ayush.bronze.reefer_telemetry

UNION ALL

SELECT 
    'wms_scan_events' AS feed,
    COUNT(*) AS rows,
    COUNT(DISTINCT _source_file) AS files
FROM aistra_ayush.bronze.wms_scan_events

UNION ALL

SELECT 
    'erp_outlet_master' AS feed,
    COUNT(*) AS rows,
    COUNT(DISTINCT _source_file) AS files
FROM aistra_ayush.bronze.erp_outlet_master

UNION ALL

SELECT 
    'erp_product_master' AS feed,
    COUNT(*) AS rows,
    COUNT(DISTINCT _source_file) AS files
FROM aistra_ayush.bronze.erp_product_master

UNION ALL

SELECT 
    'erp_sales_order_header' AS feed,
    COUNT(*) AS rows,
    COUNT(DISTINCT _source_file) AS files
FROM aistra_ayush.bronze.erp_sales_order_header

ORDER BY feed;